# Practice: concurrency

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-08-practice-concurrency.ipynb)

Companion to the [Concurrency](../learning/python-08-concurrency.ipynb) chapter. The exercises use thread pools, locks, queues and `asyncio`. Each async check is awaited at the top level of its cell, which notebooks support.

<details>
<summary><b>How to use this notebook</b></summary>

Work through the exercises in order. Each one has three parts:

1. **Question:** what to build, with the rules it must follow.
2. **Exercise cell:** replace the `pass` with your code. Keep the function name and parameters.
3. **Check cell:** run it after your code. Each line says `pass` when your function gives the expected result, and `todo` when it does not yet. Checks never stop the notebook, so you can run everything at any time.

The last cell prints your score. Use `Runtime -> Run all` to rerun everything after a change.

</details>

In [ ]:
_tally = {"passed": 0, "total": 0}

def check(label, make_actual, expected):
    """Run one check. It never raises, so the notebook always runs from top to bottom."""
    _tally["total"] += 1
    try:
        actual = make_actual()
    except Exception as error:
        print(f"error  {label}: {type(error).__name__}: {error}")
        return
    if actual == expected:
        _tally["passed"] += 1
        print(f"pass   {label}")
    else:
        print(f"todo   {label}: got {actual!r}")

def raises(make_actual, error_type):
    """True when the call raises error_type, False when it raises anything else or nothing."""
    try:
        make_actual()
    except error_type:
        return True
    except Exception:
        return False
    return False

def score():
    print(f"\nScore: {_tally['passed']} of {_tally['total']} checks pass")

In [ ]:
async def check_async(label, make_actual, expected):
    """The async version of check: it awaits the call and never raises."""
    _tally["total"] += 1
    try:
        actual = await make_actual()
    except Exception as error:
        print(f"error  {label}: {type(error).__name__}: {error}")
        return
    if actual == expected:
        _tally["passed"] += 1
        print(f"pass   {label}")
    else:
        print(f"todo   {label}: got {actual!r}")

## Exercise 1: map in threads, keeping the input order

Write `parallel_map(fn, items)`. Run `fn` on every item using a `ThreadPoolExecutor` with four workers, and return the results as a list in the same order as `items`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def parallel_map(fn, items):
    # your code here
    pass

In [ ]:
check("parallel_map squares", lambda: parallel_map(lambda x: x * x, [1, 2, 3]), [1, 4, 9])
check("parallel_map keeps order", lambda: parallel_map(str.upper, ["a", "b"]), ["A", "B"])

<details>
<summary><b>Show solution</b></summary>

```python
from concurrent.futures import ThreadPoolExecutor

def parallel_map(fn, items):
    with ThreadPoolExecutor(max_workers=4) as pool:
        return list(pool.map(fn, items))
```

</details>

## Exercise 2: a counter that threads update safely

Write `count_with_lock(workers, increments)`. Start `workers` threads. Each thread adds 1 to a shared counter `increments` times, and holds a `threading.Lock` for each addition. Return the final total.

In [ ]:
import threading

def count_with_lock(workers, increments):
    # your code here
    pass

In [ ]:
check("count_with_lock(4, 1000) == 4000", lambda: count_with_lock(4, 1000), 4000)
check("count_with_lock(1, 5) == 5", lambda: count_with_lock(1, 5), 5)

<details>
<summary><b>Show solution</b></summary>

```python
import threading

def count_with_lock(workers, increments):
    lock = threading.Lock()
    total = [0]

    def bump():
        for _ in range(increments):
            with lock:
                total[0] += 1

    threads = [threading.Thread(target=bump) for _ in range(workers)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    return total[0]
```

</details>

## Exercise 3: sum numbers with a queue and worker threads

Write `sum_with_queue(numbers, workers)`. Put every number on a `queue.Queue`. Start `workers` threads that take numbers off the queue and store them in a shared list, under a lock. Stop each worker with a `None` sentinel. Return the sum of the stored numbers. An empty input gives `0`.

*Hint: put one `None` on the queue for each worker, after all the numbers.*

In [ ]:
import queue
import threading

def sum_with_queue(numbers, workers):
    # your code here
    pass

In [ ]:
check("sum_with_queue(range(10), 3) == 45", lambda: sum_with_queue(range(10), 3), 45)
check("sum_with_queue([], 2) == 0", lambda: sum_with_queue([], 2), 0)

<details>
<summary><b>Show solution</b></summary>

```python
import queue
import threading

def sum_with_queue(numbers, workers):
    tasks = queue.Queue()
    stored = []
    lock = threading.Lock()

    def worker():
        while True:
            item = tasks.get()
            if item is None:
                return
            with lock:
                stored.append(item)

    threads = [threading.Thread(target=worker) for _ in range(workers)]
    for thread in threads:
        thread.start()
    for number in numbers:
        tasks.put(number)
    for _ in threads:
        tasks.put(None)
    for thread in threads:
        thread.join()
    return sum(stored)
```

</details>

## Exercise 4: run coroutines with a concurrency limit

Write `async def gather_limited(fn, items, limit)`. `fn` is an async function. Run `fn(item)` for every item at the same time, but with at most `limit` calls in progress at once. Return the results in the input order.

*Hint: an `asyncio.Semaphore` with `async with` limits the calls, and `asyncio.gather` keeps the order.*

In [ ]:
import asyncio

async def gather_limited(fn, items, limit):
    # your code here
    pass

In [ ]:
async def double(x):
    await asyncio.sleep(0)
    return 2 * x

await check_async("gather_limited doubles in order", lambda: gather_limited(double, [1, 2, 3], 2), [2, 4, 6])

<details>
<summary><b>Show solution</b></summary>

```python
import asyncio

async def gather_limited(fn, items, limit):
    gate = asyncio.Semaphore(limit)

    async def run(item):
        async with gate:
            return await fn(item)

    return await asyncio.gather(*(run(item) for item in items))
```

</details>

## Exercise 5: keep errors as results

`divide` is given below. Write `async def divide_all(pairs)`. It calls `divide(a, b)` for every pair at the same time. Use `return_exceptions=True`, so that a failing pair gives its exception object in the results instead of stopping the whole batch.

In [ ]:
import asyncio

async def divide(a, b):
    await asyncio.sleep(0)
    return a / b

async def divide_all(pairs):
    # your code here
    pass

In [ ]:
async def divide_all_names():
    results = await divide_all([(4, 2), (1, 0)])
    return [type(r).__name__ if isinstance(r, Exception) else r for r in results]

await check_async("divide_all keeps the ZeroDivisionError", divide_all_names, [2.0, "ZeroDivisionError"])

<details>
<summary><b>Show solution</b></summary>

```python
import asyncio

async def divide(a, b):
    await asyncio.sleep(0)
    return a / b

async def divide_all(pairs):
    return await asyncio.gather(*(divide(a, b) for a, b in pairs), return_exceptions=True)
```

</details>

## Exercise 6: a result or `None` after a timeout

Write `async def first_result_or_none(make_coro, seconds)`. `make_coro` is a function that returns a coroutine. Wait for that coroutine for at most `seconds`. Return its result, or `None` when the wait times out.

*Hint: `asyncio.wait_for` raises `asyncio.TimeoutError` when time runs out.*

In [ ]:
import asyncio

async def first_result_or_none(make_coro, seconds):
    # your code here
    pass

In [ ]:
async def fast():
    return "early"

async def slow():
    await asyncio.sleep(1)
    return "late"

async def both_results():
    return (await first_result_or_none(fast, 1), await first_result_or_none(slow, 0.01))

await check_async("fast returns its result, slow returns None", both_results, ("early", None))

<details>
<summary><b>Show solution</b></summary>

```python
import asyncio

async def first_result_or_none(make_coro, seconds):
    try:
        return await asyncio.wait_for(make_coro(), seconds)
    except asyncio.TimeoutError:
        return None
```

</details>

## Score

In [ ]:
score()